# 🏥 HealthGrid DocBot: 8-Billion Parameter Clinical LLM Fine-Tuning Pipeline
### Train Meta Llama-3.1-8B-Instruct with QLoRA & Unsloth on Free Google Colab T4 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NullErrOR-404/Health-Grid/blob/main/notebooks/HealthGrid_Llama3_8B_Clinical_FineTuning.ipynb)

This production-ready notebook fine-tunes **Meta Llama-3.1-8B-Instruct** (or **Qwen-2.5-7B-Instruct**) into an autonomous clinical physician consultant for **HealthGrid**.

**Key Pillars of HealthGrid Fine-Tuning:**
1. **Medical Chain-of-Thought (`<thought>`)**: Diagnostic differential deduction before concluding.
2. **Jan Aushadhi (PMBJP) Generic Formulary**: Seamlessly quotes authentic generic drug prices with 50%-90% savings.
3. **Zero-Trust Privacy (DPDP Act 2023)**: Defends against authority impersonation attempts.
4. **Emergency 108 Triage**: Recognizes acute red flags (ACS, stroke, severe respiratory distress) and triggers level-1 emergency protocols.
5. **100% Free Cloud Serving**: Pushes the fine-tuned adapter to Hugging Face Hub, enabling instant serverless inference via `https://router.huggingface.co` without paying for GPU servers.

## ⚙️ Step 1: Install Unsloth AI & Accelerators
Unsloth AI accelerates fine-tuning by **2x-5x** and cuts VRAM usage by **70%**, enabling an 8-billion parameter model to easily fit within a standard free **NVIDIA T4 GPU** (15GB VRAM).

In [ ]:
# Install Unsloth AI and PyTorch dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets transformers huggingface_hub

## 🧠 Step 2: Load Meta Llama-3.1-8B-Instruct (4-bit NF4 Quantization)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Supports RoPE scaling up to 8192 if needed
dtype = None # Auto detection: Float16 for T4, Bfloat16 for Ampere+
load_in_4bit = True # 4-bit NormalFloat (NF4) quantization saves 75% memory

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print(f"[*] Loading base model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("[+] Model and Tokenizer successfully loaded in 4-bit precision!")

## 🎯 Step 3: Setup Parameter-Efficient QLoRA Adapters
We inject LoRA trainable rank adapters ($r=16$, $\alpha=32$) across all attention projections and MLP feedforward gates.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves 30% more VRAM
    random_state = 3407,
    max_seq_length = max_seq_length,
)
print("[+] QLoRA adapters successfully injected! Only ~0.5% parameters are trainable.")

## 📂 Step 4: Download & Format HealthGrid Multi-Task Clinical Dataset

In [ ]:
import os
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Set chat template to Llama-3.1 standard
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# Download HealthGrid training dataset directly from GitHub repository
dataset_url = "https://raw.githubusercontent.com/NullErrOR-404/Health-Grid/main/data/healthgrid_clinical_agent_train.jsonl"
!wget -O healthgrid_train.jsonl {dataset_url}

# Load dataset
dataset = load_dataset("json", data_files="healthgrid_train.jsonl", split="train")
print(f"[*] Loaded {len(dataset)} verified clinical training samples.")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("[+] Sample processed prompt preview:")
print(dataset[0]["text"][:600] + "...")

## 🚀 Step 5: Execute Supervised Fine-Tuning (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120, # ~2-3 epochs on dataset
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("[*] Commencing QLoRA training run...")
trainer_stats = trainer.train()
print("[+] Training completed successfully!")

## 🧪 Step 6: Test Clinical Inference & Diagnostic Acumen

In [ ]:
# Enable native fast inference in Unsloth
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are HealthGrid DocBot, an autonomous chief medical AI consultant. Conduct structured differential reasoning before advising."},
    {"role": "user", "content": "Doctor, I have had acute burning epigastric stomach pain that feels like heartburn after eating for 2 weeks. I spent 180 rupees on Pantocid 40. Are there generic alternatives and what should I do?"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True, temperature = 0.3)
response = tokenizer.batch_decode(outputs)
print(response[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1])

## ☁️ Step 7: Push to Hugging Face Hub (Enables Free Serverless Cloud API)
By pushing the trained adapter to your **Hugging Face Hub** account, Hugging Face automatically hosts it on their **Free Serverless Inference API**!
The deployed HealthGrid website (`healthgrid-app.vercel.app`) can immediately call it via `https://router.huggingface.co/hf-inference/models/<your-username>/healthgrid-clinical-llama-3.1-8b`.

In [ ]:
from huggingface_hub import login

# Insert your Hugging Face Access Token with WRITE permissions (from https://huggingface.co/settings/tokens)
HF_TOKEN = "your_huggingface_write_token_here"
HUB_MODEL_ID = "your_username/healthgrid-clinical-llama-3.1-8b"

login(token=HF_TOKEN)

# Option A: Push the compact LoRA Adapter (~150MB, fast and free)
print(f"[*] Uploading LoRA adapter to Hugging Face Hub: {HUB_MODEL_ID}-adapter...")
model.push_to_hub_lora(f"{HUB_MODEL_ID}-adapter", tokenizer, token=HF_TOKEN)

# Option B: Push the merged 16-bit Full Model for direct serverless inference
print(f"[*] Uploading merged 16-bit model to Hugging Face Hub: {HUB_MODEL_ID}...")
model.push_to_hub_merged(HUB_MODEL_ID, tokenizer, save_method="merged_16bit", token=HF_TOKEN)

# Option C: Export 4-bit GGUF for local Ollama running
print("[*] Exporting Q4_K_M GGUF file for local Ollama running...")
model.save_pretrained_gguf("healthgrid-docbot-8b-gguf", tokenizer, quantization_method="q4_k_m")

print("\n[🎉 SUCCESS] Model uploaded to Hugging Face Hub!")
print(f"[*] Free Serverless API Endpoint: https://router.huggingface.co/hf-inference/models/{HUB_MODEL_ID}")
print("[*] Update your HealthGrid frontend/.env with:")
print(f"    VITE_HF_FINE_TUNED_MODEL={HUB_MODEL_ID}")